# Task 3 — Saved gender model diagnostic

This notebook **does not train**. It reloads saved G2 and CompactBlurCNN models for folds 0 and 4, then checks clean training and validation images. Use a Colab GPU and Run All. Keep the held-out test sealed.

The new source module must be on the selected repository branch before running this notebook.


## 1. Connect Drive and load the repository


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import time
import zipfile

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "task-3-gender-usage-classification"
REPO_DIR = Path("/content/MLA2")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT / "MyDrive/MLA2"
DATA_ZIP = DRIVE_PROJECT_DIR / "data/task3-data.zip"
LOCAL_DATA_ZIP = Path("/content/task3-data.zip")
DRIVE_TASK_DIR = DRIVE_PROJECT_DIR / "task3"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "results/runs.csv"
LOCAL_REGISTRY = REPO_DIR / "results/runs.csv"

def run_checked(command, *, cwd=None):
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Connect this notebook to a Google Colab GPU runtime first.") from exc

drive.mount(str(DRIVE_MOUNT), force_remount=False)
if (REPO_DIR / ".git").is_dir():
    remote_url = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True
    ).strip()
    if remote_url != REPO_URL:
        raise RuntimeError(f"{REPO_DIR} belongs to a different repository: {remote_url}")
    run_checked(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "switch", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=REPO_DIR)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
else:
    run_checked(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR])

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("Repository ready:", REPO_DIR)
print("Commit:", commit)


## 2. Restore images and the canonical split


In [ ]:
def copy_teacher_zip_to_local_disk():
    if LOCAL_DATA_ZIP.is_file():
        try:
            with zipfile.ZipFile(LOCAL_DATA_ZIP) as existing:
                existing.infolist()
            return
        except zipfile.BadZipFile:
            LOCAL_DATA_ZIP.unlink()
    partial = LOCAL_DATA_ZIP.with_suffix(".zip.partial")
    for attempt in range(1, 4):
        partial.unlink(missing_ok=True)
        try:
            expected_bytes = DATA_ZIP.stat().st_size
            with DATA_ZIP.open("rb") as source, partial.open("wb") as target:
                shutil.copyfileobj(source, target, length=8 * 1024**2)
            if partial.stat().st_size != expected_bytes:
                raise OSError("The local ZIP copy is incomplete.")
            partial.replace(LOCAL_DATA_ZIP)
            return
        except OSError as error:
            partial.unlink(missing_ok=True)
            if attempt == 3:
                raise RuntimeError("Drive disconnected three times. Remount and retry.") from error
            drive.mount(str(DRIVE_MOUNT), force_remount=True)
            time.sleep(2)

copy_teacher_zip_to_local_disk()
teacher_dir = REPO_DIR / "data/raw/teacher"
required_files = (
    teacher_dir / "train/styles_train.csv",
    teacher_dir / "test/styles_prediction.csv",
)
image_dirs = (teacher_dir / "train/images_train", teacher_dir / "test/images_test")
image_suffixes = {".jpg", ".jpeg"}
with zipfile.ZipFile(LOCAL_DATA_ZIP) as archive:
    names = archive.namelist()
    if any(Path(name).is_absolute() or ".." in Path(name).parts for name in names):
        raise RuntimeError("The teacher archive contains an unsafe path.")
    expected_images = sum(
        name.startswith("data/raw/teacher/") and Path(name).suffix.lower() in image_suffixes
        for name in names
    )
    current_images = sum(
        path.suffix.lower() in image_suffixes for folder in image_dirs for path in folder.glob("*")
    )
    if current_images != expected_images or not all(path.is_file() for path in required_files):
        archive.extractall(REPO_DIR)

actual_images = sum(
    path.suffix.lower() in image_suffixes for folder in image_dirs for path in folder.glob("*")
)
if actual_images != expected_images or not all(path.is_file() for path in required_files):
    raise RuntimeError(f"Teacher data is incomplete: {actual_images:,}/{expected_images:,} images")

os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
(DRIVE_TASK_DIR / "results").mkdir(parents=True, exist_ok=True)
print(f"Teacher data ready: {actual_images:,} images")


## 3. Run saved-model checks

Verify each development image against its SHA-256 content hash in the canonical split. Then verify checkpoint hashes, configuration, classes and split before loading weights. Reproduce clean training F1 and saved validation probabilities. Save clean predictions and class, article-type and family-size slices. Family size means the number of usable gender images in that product family across development data.

For each model and partition, sample up to 32 images per class with seed 2753. Compare batches of 1, 32 and 128 in forward, reverse and shuffled order. Flag probability changes above 0.00001 or any changed predicted label. Verify that weights and BatchNorm buffers stay unchanged. These tolerances detect a need for review; they are not model-selection rules.

Outputs use a new timestamped Drive folder. Source checkpoints and the run registry are only read.


In [ ]:
from datetime import datetime, timezone
from fashion.train.task3_gender_diagnostic import run_gender_diagnostic

OUTPUT = DRIVE_TASK_DIR / "diagnostics/gender_generalization" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
result = run_gender_diagnostic(
    g2_directory=DRIVE_TASK_DIR / "experiments/t3_gender_v2_g2_translation/gender",
    compact_directory=DRIVE_TASK_DIR / "experiments/t3_gender_e5_compact_blur_cnn/gender",
    registry_path=DRIVE_REGISTRY, output=OUTPUT, root=REPO_DIR,
)
print("Diagnostic:", result["status"])
print("Saved to:", OUTPUT)
for run in result["runs"]:
    print(run["model"], "fold", run["fold"], "checks passed:", run["pass"])


## 4. Stop and review

Tell me when this finishes. I can read the saved Drive results and consult the other task. Do not start another training run from these diagnostic checks alone.

Read `diagnostic_status.json`, `clean_class_scores.csv`, `clean_slices.csv`, `slice_accuracy_gaps.csv` and `batch_order_stability.csv`. Each model folder contains clean per-image predictions and fixed sample IDs. Slice scores are accuracy (fraction correct); overall reproduced scores are macro-F1 (F1 averaged across classes). Small slices and reused development folds limit what we can conclude.
